# TNBC Drug Cytotoxicity Prediction — Training Pipeline

This notebook trains a three-phase transfer learning model for predicting drug cytotoxicity (LN_IC50) in Triple-Negative Breast Cancer (TNBC) cell lines. A hybrid GIN + TransformerConv GNN encodes drug molecular structures, while pathway scores (GSVA), somatic mutations, and proteomics encode cell line characteristics.

**Pipeline overview:**
1. **Phase 1** — Pan-cancer pre-training on GDSC2 (all TNBC cells excluded)
2. **Phase 2** — Breast cancer fine-tuning (all TNBC cells excluded)
3. **Phase 3** — TNBC-specific fine-tuning with Leave-One-Cell-Line-Out (LOCO) CV (21 folds)

All splits are cell-line-based to prevent data leakage. Evaluation, baselines, and figures are in `visualisations.ipynb`.


## 1. Setup and Imports


In [ ]:
import json
import os
import pickle
import shutil
import time
import warnings

import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from scipy.stats import pearsonr, spearmanr, t
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, OneCycleLR
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch
from torch_geometric.nn import TransformerConv, GINConv, global_mean_pool, global_max_pool
from torch_scatter import scatter_softmax

from rdkit import Chem
from rdkit.Chem import AllChem
from tqdm import tqdm

warnings.filterwarnings('ignore')

current_dir = Path.cwd()
project_root = current_dir

if (current_dir / "data" / "raw").exists():
    project_root = current_dir
elif (current_dir.parent / "data" / "raw").exists():
    project_root = current_dir.parent
elif (current_dir.parent.parent / "data" / "raw").exists():
    project_root = current_dir.parent.parent

data_dir = project_root / "data" / "raw"
output_dir = project_root / "results"
models_dir = project_root / "models"
splits_dir = project_root / "data_splits"
prebatched_dir = project_root / "prebatched_data"

for dir_path in [output_dir, models_dir, splits_dir, prebatched_dir]:
    dir_path.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

## 2. Data Loading and Preprocessing

Load GDSC2 drug response data, GSVA pathway scores, cell line metadata, somatic mutations, and proteomics. Map all data sources to DepMap `ModelID` identifiers and merge into a single pan-cancer DataFrame.


In [ ]:
gdsc2_df = pd.read_excel(data_dir / "GDSC2 Fitted Dose Response Oct 27 2023.xlsx")
model_df = pd.read_csv(data_dir / "DepMap Model Data.csv")
drug_smiles = pd.read_csv(data_dir / "drugs_with_smiles.csv")
gsva_path = data_dir / "cell_ge.csv"

pathway_scores_raw = pd.read_csv(gsva_path, index_col=0)

pathway_names = pathway_scores_raw.index.tolist()
pathway_data = pathway_scores_raw.T.reset_index()
pathway_data.columns = ['CellLineName'] + pathway_names

rmse_col = [col for col in gdsc2_df.columns if 'RMSE' in col.upper()][0]
gdsc2_filtered = gdsc2_df[gdsc2_df[rmse_col] < 0.3].copy()

drug_response = gdsc2_filtered[['DRUG_NAME', 'CELL_LINE_NAME', 'LN_IC50', 'COSMIC_ID']].copy()
drug_response.columns = ['DrugName', 'CellLineName', 'LN_IC50', 'COSMICID']

cell_name_to_modelid = dict(zip(
    model_df['StrippedCellLineName'].str.upper().str.replace('-', '').str.replace('_', ''),
    model_df['ModelID']
))
cosmic_to_modelid = model_df.drop_duplicates(subset='COSMICID', keep='first').set_index('COSMICID')['ModelID'].to_dict()

pathway_data['ModelID'] = pathway_data['CellLineName'].apply(
    lambda x: cell_name_to_modelid.get(str(x).upper().replace('-', '').replace('_', ''), None)
)
pathway_data = pathway_data[pathway_data['ModelID'].notna()].copy()

drug_response['ModelID'] = drug_response['COSMICID'].apply(lambda x: cosmic_to_modelid.get(x, None))
unmapped = drug_response[drug_response['ModelID'].isna()]
if len(unmapped) > 0:
    drug_response.loc[drug_response['ModelID'].isna(), 'ModelID'] = drug_response.loc[drug_response['ModelID'].isna(), 'CellLineName'].apply(
        lambda x: cell_name_to_modelid.get(str(x).upper().replace('-', '').replace('_', ''), None)
    )
drug_response = drug_response[drug_response['ModelID'].notna()].drop(columns=['COSMICID'])

pan_cancer_pathway = drug_response.merge(
    pathway_data[['ModelID'] + pathway_names],
    on='ModelID',
    how='inner'
)

drug_smiles_renamed = drug_smiles.rename(columns={'DRUG_NAME': 'DrugName'})
pan_cancer_pathway = pan_cancer_pathway.merge(
    drug_smiles_renamed[['DrugName', 'SMILES']],
    on='DrugName',
    how='inner'
)

pan_cancer_pathway = pan_cancer_pathway[pan_cancer_pathway['LN_IC50'].notna()].copy()

model_cols = ['ModelID', 'StrippedCellLineName', 'OncotreeLineage', 'OncotreePrimaryDisease', 'OncotreeSubtype']
for col in ['PatientSubtypeFeatures', 'ModelSubtypeFeatures']:
    if col in model_df.columns:
        model_cols.append(col)
pan_cancer_pathway = pan_cancer_pathway.merge(
    model_df[model_cols],
    on='ModelID',
    how='left'
)

drug_name_col = 'DrugName'
ln_ic50_col = 'LN_IC50'
pathway_cols = pathway_names
drugs_with_smiles = drug_smiles_renamed[['DrugName', 'SMILES']].copy()

print(f"Pathway data: {len(pathway_data)} cell lines | Drug response: {len(drug_response)} rows | Merged: {len(pan_cancer_pathway)} rows")

In [ ]:
mutation_file = data_dir / "OmicsSomaticMutations.csv"
if not mutation_file.exists():
    mutation_file = data_dir / "Omics Somatic Mutations.csv"

mutations_df = pd.read_csv(mutation_file, low_memory=False)

priority_genes = [
    'BRCA1', 'BRCA2', 'TP53', 'PIK3CA', 'PTEN', 'RB1',
    'EGFR', 'MYC', 'CDKN2A', 'GATA3', 'MAP3K1', 'KRAS',
    'ARID1A', 'NF1', 'ERBB2', 'ESR1', 'PGR', 'AKT1'
]

gene_col = None
if 'HugoSymbol' in mutations_df.columns:
    gene_col = 'HugoSymbol'
elif 'Hugo_Symbol' in mutations_df.columns:
    gene_col = 'Hugo_Symbol'
else:
    raise ValueError(f"Could not find gene symbol column. Available columns: {list(mutations_df.columns)[:20]}")

# Determine deleterious mutation status using best available indicator
if 'LikelyLoF' in mutations_df.columns:
    mutations_df['is_mutated'] = mutations_df['LikelyLoF'].fillna(False).astype(int)
elif 'TranscriptLikelyLof' in mutations_df.columns:
    mutations_df['is_mutated'] = mutations_df['TranscriptLikelyLof'].fillna(False).astype(int)
elif 'VepImpact' in mutations_df.columns:
    mutations_df['is_mutated'] = mutations_df['VepImpact'].isin(['HIGH', 'MODERATE']).astype(int)
elif 'MolecularConsequence' in mutations_df.columns:
    non_synonymous = ['missense_variant', 'nonsense_variant', 'frameshift_variant',
                      'splice_acceptor_variant', 'splice_donor_variant', 'stop_gained',
                      'stop_lost', 'start_lost', 'inframe_insertion', 'inframe_deletion']
    mutations_df['is_mutated'] = mutations_df['MolecularConsequence'].isin(non_synonymous).astype(int)
else:
    mutations_df['is_mutated'] = 1

available_priority_genes = [gene for gene in priority_genes if gene in mutations_df[gene_col].dropna().values]
if len(available_priority_genes) > 0:
    mutations_filtered = mutations_df[mutations_df[gene_col].isin(available_priority_genes)].copy()
else:
    mutations_filtered = mutations_df.copy()
    available_priority_genes = mutations_df[gene_col].dropna().unique().tolist()

mutation_pivot = mutations_filtered.groupby(['ModelID', gene_col])['is_mutated'].max().unstack(fill_value=0)
mutation_pivot = mutation_pivot.astype(int)
mutation_pivot['total_mutation_burden'] = mutation_pivot.sum(axis=1)
mutation_features = mutation_pivot.reset_index()
mutation_cols = [col for col in mutation_features.columns if col != 'ModelID']

pathway_data = pathway_data.merge(
    mutation_features[['ModelID'] + mutation_cols],
    on='ModelID',
    how='left'
).fillna(0)

pan_cancer_pathway = pan_cancer_pathway.merge(
    mutation_features[['ModelID'] + mutation_cols],
    on='ModelID',
    how='left'
).fillna(0)

actual_pathway_count = len(pathway_cols)
actual_mutation_count = len(mutation_cols)
total_cell_input_dim = actual_pathway_count + actual_mutation_count

print(f"Mutations: {actual_mutation_count} features | Total cell input: {total_cell_input_dim} ({actual_pathway_count} pathways + {actual_mutation_count} mutations)")

In [ ]:
proteomics_path = data_dir / "Breast Cancer Proteomic Dynamics (2).csv"
try:
    proteomics_raw = pd.read_csv(proteomics_path, sep=';')
except Exception:
    proteomics_raw = None

if proteomics_raw is not None:
    protein_ids = proteomics_raw.iloc[:, 0].astype(str) + ';' + proteomics_raw.iloc[:, 1].astype(str)
    proteomics_raw = proteomics_raw.set_index(protein_ids)
    proteomics_raw = proteomics_raw.iloc[:, 2:]

    def _to_float(val):
        if pd.isna(val) or val == '':
            return np.nan
        s = str(val).strip().replace(',', '.')
        try:
            return float(s)
        except ValueError:
            return np.nan

    for c in proteomics_raw.columns:
        proteomics_raw[c] = proteomics_raw[c].apply(_to_float)

    proteomics_T = proteomics_raw.T
    def cell_col_to_base(name):
        base = str(name).split('_')[0] if '_' in str(name) else str(name)
        return base.upper().replace('-', '').replace(' ', '')

    cell_to_base = {c: cell_col_to_base(c) for c in proteomics_T.index}
    proteomics_T.index = proteomics_T.index.map(lambda x: cell_name_to_modelid.get(cell_to_base[x], None))
    proteomics_T = proteomics_T[proteomics_T.index.notna()]
    proteomics_T.index = proteomics_T.index.astype(str)
    proteomics_by_modelid = proteomics_T.groupby(proteomics_T.index).median()
    assert proteomics_by_modelid.index.is_unique, \
        f"Duplicate ModelIDs after groupby: {proteomics_by_modelid.index[proteomics_by_modelid.index.duplicated()].tolist()}"
    proteomics_by_modelid = proteomics_by_modelid.loc[:, ~proteomics_by_modelid.columns.duplicated()]
    frac_observed = proteomics_by_modelid.notna().mean(axis=0)
    protein_cols = proteomics_by_modelid.columns[frac_observed >= 0.70].tolist()
    protein_cols = list(dict.fromkeys(protein_cols))  # deduplicate while preserving order
    proteomics_by_modelid = proteomics_by_modelid[protein_cols]
    proteomics_by_modelid_full = proteomics_by_modelid.copy()
    protein_cols_full = protein_cols.copy()
    n_proteins_full = len(protein_cols_full)
    protein_cols = protein_cols_full
    n_proteins = n_proteins_full
else:
    proteomics_by_modelid = pd.DataFrame()
    proteomics_by_modelid_full = pd.DataFrame()
    protein_cols = []
    protein_cols_full = []
    n_proteins = 0
    n_proteins_full = 0

print(f"Proteomics: {n_proteins} proteins, {len(proteomics_by_modelid)} cell lines")
if n_proteins > 0:
    prot_ids = set(proteomics_by_modelid.index)
    gdsc_ids = set(pan_cancer_pathway['ModelID'].astype(str))
    lineage_mask = pan_cancer_pathway['OncotreeLineage'] == 'Breast'
    lineage_ids = set(pan_cancer_pathway.loc[lineage_mask, 'ModelID'].astype(str))
    b = pan_cancer_pathway.loc[lineage_mask]
    _sub_col = 'ModelSubtypeFeatures'
    _sub_kw = 'TNBC'
    if _sub_col in b.columns:
        sub_mask = b[_sub_col].astype(str).str.contains(_sub_kw, case=False, na=False)
    else:
        psf = b['PatientSubtypeFeatures'].astype(str).str.contains(_sub_kw, case=False, na=False) if 'PatientSubtypeFeatures' in b.columns else pd.Series(False, index=b.index)
        msf = b['ModelSubtypeFeatures'].astype(str).str.contains(_sub_kw, case=False, na=False) if 'ModelSubtypeFeatures' in b.columns else pd.Series(False, index=b.index)
        sub_mask = psf | msf
    subtype_ids = set(b.loc[sub_mask, 'ModelID'].astype(str))
    print(f"Proteomics overlap: {len(prot_ids & gdsc_ids)} GDSC, {len(prot_ids & lineage_ids)} Breast, {len(prot_ids & subtype_ids)} TNBC")

### 2.1 Drug Molecular Graph Construction

Convert SMILES strings to PyTorch Geometric graphs via RDKit. Atom features (9-dim): atomic number, degree, formal charge, hybridisation, aromaticity, total Hs, radical electrons, ring membership, chirality. Bond features (4-dim): bond type, conjugation, ring membership, stereo.


In [ ]:
def get_atom_features(atom):
    """
    Extract atom features for GNN.

    Args:
        atom: RDKit atom object

    Returns:
        List of atom features
    """
    features = [
        atom.GetAtomicNum(),
        atom.GetDegree(),
        atom.GetFormalCharge(),
        atom.GetHybridization().real,
        atom.GetIsAromatic(),
        atom.GetTotalNumHs(),
        atom.GetNumRadicalElectrons(),
        atom.IsInRing(),
        atom.GetChiralTag().real,
    ]
    return features

def get_bond_features(bond):
    """
    Extract bond features for GNN.

    Args:
        bond: RDKit bond object

    Returns:
        List of bond features
    """
    features = [
        bond.GetBondTypeAsDouble(),
        bond.GetIsConjugated(),
        bond.IsInRing(),
        bond.GetStereo().real,
    ]
    return features

def smiles_to_graph(smiles_string):
    """
    Convert SMILES string to PyTorch Geometric graph.

    Args:
        smiles_string: SMILES representation of molecule

    Returns:
        torch_geometric.data.Data object or None if invalid
    """
    try:
        mol = Chem.MolFromSmiles(smiles_string)
        if mol is None:
            return None

        mol = Chem.AddHs(mol)

        node_features = []
        for atom in mol.GetAtoms():
            node_features.append(get_atom_features(atom))

        if len(node_features) == 0:
            return None

        x = torch.tensor(node_features, dtype=torch.float)

        edge_indices = []
        edge_features = []

        for bond in mol.GetBonds():
            i = bond.GetBeginAtomIdx()
            j = bond.GetEndAtomIdx()

            edge_indices.append([i, j])
            edge_indices.append([j, i])

            bond_feats = get_bond_features(bond)
            edge_features.append(bond_feats)
            edge_features.append(bond_feats)

        if len(edge_indices) == 0:
            edge_indices = [[0, 0]]
            edge_features = [[0.0] * 4]

        edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_features, dtype=torch.float)

        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

    except Exception:
        return None

In [ ]:
def preprocess_drugs(drugs_df, drug_name_col, smiles_col, save_path):
    """
    Convert all drugs to graphs and cache.

    Args:
        drugs_df: DataFrame with drug names and SMILES
        drug_name_col: Column name for drug names
        smiles_col: Column name for SMILES strings
        save_path: Path to save cached graphs

    Returns:
        Dictionary mapping drug names to graph data
    """
    drug_graphs = {}
    failed_drugs = []

    for idx, row in tqdm(drugs_df.iterrows(), total=len(drugs_df), desc="Converting SMILES"):
        drug_name = row[drug_name_col]
        smiles = row[smiles_col]

        graph = smiles_to_graph(smiles)

        if graph is not None:
            drug_graphs[drug_name] = {
                'drug_name': drug_name,
                'smiles': smiles,
                'graph_data': graph,
                'node_dim': graph.x.shape[1],
                'edge_dim': graph.edge_attr.shape[1] if graph.edge_attr is not None else 0,
                'num_atoms': graph.x.shape[0],
                'num_bonds': graph.edge_index.shape[1]
            }
        else:
            failed_drugs.append(drug_name)

    with open(save_path, 'wb') as f:
        pickle.dump(drug_graphs, f)

    return drug_graphs

drug_graphs_path = project_root / "data" / "processed" / "drug_graphs.pkl"
drug_graphs = preprocess_drugs(drugs_with_smiles, drugs_with_smiles.columns[0], 'SMILES', drug_graphs_path)

valid_drugs = set(drug_graphs.keys())
before_count = len(pan_cancer_pathway)
before_drugs = set(pan_cancer_pathway['DrugName'].unique())
pan_cancer_pathway = pan_cancer_pathway[pan_cancer_pathway['DrugName'].isin(valid_drugs)].copy().reset_index(drop=True)
after_count = len(pan_cancer_pathway)
removed_drugs = before_drugs - valid_drugs
if len(removed_drugs) > 0:
    print(f"Dropped {len(removed_drugs)} drugs without valid graphs, {before_count} -> {after_count} samples")

### 2.2 Cancer Subtype Filtering

In [ ]:
breast_pathway = pan_cancer_pathway[
    pan_cancer_pathway['OncotreeLineage'] == 'Breast'
].copy()

_sub_col = 'ModelSubtypeFeatures'
_sub_kw  = 'TNBC'

if _sub_col in breast_pathway.columns:
    subtype_mask = breast_pathway[_sub_col].astype(str).str.contains(_sub_kw, case=False, na=False)
else:
    psf = breast_pathway['PatientSubtypeFeatures'].astype(str).str.contains(_sub_kw, case=False, na=False) \
          if 'PatientSubtypeFeatures' in breast_pathway.columns \
          else pd.Series(False, index=breast_pathway.index)
    msf = breast_pathway['ModelSubtypeFeatures'].astype(str).str.contains(_sub_kw, case=False, na=False) \
          if 'ModelSubtypeFeatures' in breast_pathway.columns \
          else pd.Series(False, index=breast_pathway.index)
    subtype_mask = psf | msf

tnbc_pathway = breast_pathway[subtype_mask].copy()

print(f"Breast: {breast_pathway['ModelID'].nunique()} cell lines, {len(breast_pathway)} samples")
print(f"TNBC: {tnbc_pathway['ModelID'].nunique()} cell lines, {len(tnbc_pathway)} samples")

if tnbc_pathway['ModelID'].nunique() < 10:
    print(f"WARNING: only {tnbc_pathway['ModelID'].nunique()} TNBC cell lines")

## 3. Model Architecture

Dataset class, GNN drug encoder, and full prediction model.

### 3.1 Dataset Class

In [ ]:
class DrugResponsePathwayDataset(Dataset):
    """
    Dataset class for drug response prediction using GSVA pathway scores.
    GSVA computes pathway enrichment scores using a non-parametric approach.
    Performs per-sample z-score normalization of pathway scores.
    """

    def __init__(self, dataframe, drug_graphs_dict, drug_col='DRUG_NAME', pathway_cols=None, mutation_cols=None,
                 proteomics_by_modelid=None, protein_cols=None, protein_mean=None, protein_std=None):
        """
        Initialize pathway-based dataset with optional mutation and proteomics features.

        Args:
            dataframe: DataFrame with pathway scores, ModelID, DRUG_NAME, LN_IC50
            drug_graphs_dict: Dictionary mapping drug names to graph data
            drug_col: Column name for drug names
            pathway_cols: List of pathway column names (if None, will infer)
            mutation_cols: List of mutation column names (if None, no mutations used)
            proteomics_by_modelid: DataFrame indexed by ModelID with protein columns (optional)
            protein_cols: List of protein column names (required if proteomics_by_modelid given)
            protein_mean: Per-protein mean for z-scoring (from train; optional)
            protein_std: Per-protein std for z-scoring (from train; optional)
        """
        self.original_data = dataframe.copy()
        self.data = dataframe.reset_index(drop=True)
        self.drug_graphs = drug_graphs_dict
        self.drug_col = drug_col

        # Proteomics
        self.proteomics_by_modelid = proteomics_by_modelid
        self.protein_cols = protein_cols if protein_cols is not None else []
        self.protein_mean = np.array(protein_mean, dtype=np.float32) if protein_mean is not None else None
        self.protein_std = np.array(protein_std, dtype=np.float32) if protein_std is not None else None
        self.n_proteins = len(self.protein_cols)

        # Identify pathway columns
        if pathway_cols is not None:
            self.pathway_cols = [c for c in pathway_cols if c in dataframe.columns]
        else:
            # Infer pathway columns: exclude metadata columns
            exclude_cols = ['ModelID', 'COSMICID', 'StrippedCellLineName', 'OncotreeLineage',
                           'OncotreePrimaryDisease', drug_col, 'SMILES', 'LN_IC50', 'CellLineName']
            # Only include columns that are numeric and not in exclude list
            numeric_cols = dataframe.select_dtypes(include=[np.number]).columns.tolist()
            self.pathway_cols = [c for c in numeric_cols if c not in exclude_cols]

        # Verify pathway columns are numeric
        for col in self.pathway_cols:
            if not pd.api.types.is_numeric_dtype(dataframe[col]):
                raise ValueError(f"Pathway column '{col}' is not numeric")

        # Handle mutation columns
        if mutation_cols is not None:
            self.mutation_cols = [c for c in mutation_cols if c in dataframe.columns]
        else:
            self.mutation_cols = []

        # Create combined feature columns list (excludes proteins - handled separately)
        self.all_feature_cols = self.pathway_cols + self.mutation_cols

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        drug_name = row[self.drug_col]

        if drug_name not in self.drug_graphs:
            raise ValueError(f"Drug {drug_name} not found in drug_graphs")
        drug_graph = self.drug_graphs[drug_name]['graph_data']

        # Extract pathway scores
        pathway_scores = row[self.pathway_cols].values.astype(np.float32)

        # Per-sample z-score normalization for pathways ONLY
        mean = pathway_scores.mean()
        std = pathway_scores.std()
        if std > 1e-8:
            pathway_scores = (pathway_scores - mean) / std
        else:
            pathway_scores = np.zeros_like(pathway_scores)

        # Extract mutation features (no normalization - already binary/count)
        if len(self.mutation_cols) > 0:
            mutation_features = row[self.mutation_cols].values.astype(np.float32)
            cell_features = np.concatenate([pathway_scores, mutation_features])
        else:
            cell_features = pathway_scores

        # Proteomics: vectorized extraction (proteomics has columns=protein_cols, values already aligned)
        if self.n_proteins > 0:
            model_id = str(row['ModelID'])
            if self.proteomics_by_modelid is not None and model_id in self.proteomics_by_modelid.index:
                cell_row = self.proteomics_by_modelid.loc[model_id]
                if isinstance(cell_row, pd.DataFrame):
                    cell_row = cell_row.median(axis=0)
                vals = np.asarray(cell_row.values, dtype=np.float32)
                if len(vals) > self.n_proteins:
                    vals = vals[:self.n_proteins]
                elif len(vals) < self.n_proteins:
                    vals = np.pad(vals, (0, self.n_proteins - len(vals)), constant_values=0)
                prots = np.nan_to_num(vals, nan=0.0, posinf=0.0, neginf=0.0)
                if self.protein_mean is not None and self.protein_std is not None:
                    n = min(len(prots), len(self.protein_mean), len(self.protein_std))
                    prots = prots[:n].astype(np.float32)
                    mean_use = np.asarray(self.protein_mean[:n], dtype=np.float32)
                    std_safe = np.where(np.asarray(self.protein_std[:n]) > 1e-8, self.protein_std[:n], 1.0).astype(np.float32)
                    prots = (prots - mean_use) / std_safe
                    if n < self.n_proteins:
                        prots = np.pad(prots, (0, self.n_proteins - n), constant_values=0)
                has_proteomics = 1.0
            else:
                prots = np.zeros(self.n_proteins, dtype=np.float32)
                has_proteomics = 0.0
            proteins_tensor = torch.tensor(prots, dtype=torch.float32)
            has_proteomics_tensor = torch.tensor([has_proteomics], dtype=torch.float32)
        else:
            proteins_tensor = torch.tensor([], dtype=torch.float32)
            has_proteomics_tensor = torch.tensor([0.0], dtype=torch.float32)

        cell_expr_tensor = torch.tensor(cell_features, dtype=torch.float32)
        ic50 = torch.tensor([row['LN_IC50']], dtype=torch.float32)

        return {
            'drug_graph': drug_graph,
            'cell_expr': cell_expr_tensor,
            'proteins': proteins_tensor,
            'has_proteomics': has_proteomics_tensor,
            'ic50': ic50,
            'drug_name': drug_name,
            'cell_id': row['ModelID']
        }


### 3.2 Dataset Instantiation

In [ ]:
# Protein z-score stats: use global (all cells with proteomics) for initial full datasets
_prot_model_ids = set(proteomics_by_modelid.index) & set(pan_cancer_pathway['ModelID'].astype(str))
if len(_prot_model_ids) > 0:
    _prot_train = proteomics_by_modelid.loc[list(_prot_model_ids), protein_cols]
    protein_mean_global = _prot_train.mean(axis=0).values
    protein_std_global = _prot_train.std(axis=0).values
    protein_std_global = np.where(protein_std_global > 1e-8, protein_std_global, 1.0)
else:
    protein_mean_global = np.zeros(n_proteins)
    protein_std_global = np.ones(n_proteins)

# cell_input = pathways + mutations + [protein_proj(256) + has_proteomics(1) if proteomics]
actual_pathway_count = len(pathway_cols)
actual_mutation_count = len(mutation_cols) if 'mutation_cols' in globals() else 0
total_cell_input_dim = actual_pathway_count + actual_mutation_count

_dataset_prot_kw = dict(
    proteomics_by_modelid=proteomics_by_modelid,
    protein_cols=protein_cols,
    protein_mean=protein_mean_global,
    protein_std=protein_std_global
) if n_proteins > 0 else {}

pan_dataset_pathway = DrugResponsePathwayDataset(pan_cancer_pathway, drug_graphs, drug_col=drug_name_col, pathway_cols=pathway_cols, mutation_cols=mutation_cols, **_dataset_prot_kw)
breast_dataset_pathway = DrugResponsePathwayDataset(breast_pathway, drug_graphs, drug_col=drug_name_col, pathway_cols=pathway_cols, mutation_cols=mutation_cols, **_dataset_prot_kw)
tnbc_dataset_pathway = DrugResponsePathwayDataset(tnbc_pathway, drug_graphs, drug_col=drug_name_col, pathway_cols=pathway_cols, mutation_cols=mutation_cols, **_dataset_prot_kw)

print(f"Cell input dim: {total_cell_input_dim} ({actual_pathway_count} pathways + {actual_mutation_count} mutations)")

### 3.3 Drug Encoder

In [ ]:
class DrugEncoder(nn.Module):
    """
    Graph Neural Network encoder for molecular SMILES structures.
    Hybrid architecture: 2 GIN layers + 2 TransformerConv layers with learned weighted sum.
    """

    def __init__(self, node_feature_dim=9, edge_feature_dim=4, hidden_dim=256, dropout=0.3):
        """
        Initialize DrugEncoder with hybrid GIN + TransformerConv architecture.

        Args:
            node_feature_dim: Number of atom features (9)
            edge_feature_dim: Number of bond features (4)
            hidden_dim: Output embedding dimension (256)
            dropout: Dropout rate
        """
        super(DrugEncoder, self).__init__()

        if torch.cuda.is_available():
            self.device = torch.device('cuda')
        elif torch.backends.mps.is_available():
            self.device = torch.device('mps')
        else:
            self.device = torch.device('cpu')

        gin_nn1 = nn.Sequential(
            nn.Linear(node_feature_dim, 128),
            nn.ReLU(),
            nn.LayerNorm(128),
            nn.Dropout(dropout),
            nn.Linear(128, 128)
        )
        self.gin_conv1 = GINConv(gin_nn1, train_eps=True)
        self.gin_bn1 = nn.BatchNorm1d(128)

        gin_nn2 = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.LayerNorm(128),
            nn.Dropout(dropout),
            nn.Linear(128, 128)
        )
        self.gin_conv2 = GINConv(gin_nn2, train_eps=True)
        self.gin_bn2 = nn.BatchNorm1d(128)

        self.transformer_conv1 = TransformerConv(node_feature_dim, 128, heads=4, dropout=dropout, edge_dim=edge_feature_dim)
        self.transformer_bn1 = nn.BatchNorm1d(128 * 4)

        self.transformer_conv2 = TransformerConv(128 * 4, 128, heads=4, concat=False, dropout=dropout, edge_dim=edge_feature_dim)
        self.transformer_bn2 = nn.BatchNorm1d(128)

        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.proj = nn.Linear(128, hidden_dim)
        self.proj_bn = nn.BatchNorm1d(hidden_dim)

        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
        self.attention_weights = nn.Linear(hidden_dim, 1)

        self.to(self.device)

    def forward(self, data):
        """
        Forward pass through hybrid GIN + TransformerConv drug encoder.

        Args:
            data: PyTorch Geometric Data/Batch object

        Returns:
            Drug embeddings of shape (batch_size, 256)
        """
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch

        x = x.to(self.device, non_blocking=True)
        edge_index = edge_index.to(self.device, non_blocking=True)
        edge_attr = edge_attr.to(self.device, non_blocking=True)
        batch = batch.to(self.device, non_blocking=True)

        x_gin = self.gin_conv1(x, edge_index)
        x_gin = self.gin_bn1(x_gin)
        x_gin = self.relu(x_gin)
        x_gin = self.dropout(x_gin)

        x_gin = self.gin_conv2(x_gin, edge_index)
        x_gin = self.gin_bn2(x_gin)
        x_gin = self.relu(x_gin)

        x_transformer = self.transformer_conv1(x, edge_index, edge_attr)
        x_transformer = self.transformer_bn1(x_transformer)
        x_transformer = self.relu(x_transformer)
        x_transformer = self.dropout(x_transformer)

        x_transformer = self.transformer_conv2(x_transformer, edge_index, edge_attr)
        x_transformer = self.transformer_bn2(x_transformer)
        x_transformer = self.relu(x_transformer)

        alpha_sigmoid = torch.sigmoid(self.alpha)
        x_merged = alpha_sigmoid * x_gin + (1 - alpha_sigmoid) * x_transformer

        x = self.proj(x_merged)
        x = self.proj_bn(x)
        x = self.relu(x)

        attention_scores = self.attention_weights(x)
        attention_scores = scatter_softmax(attention_scores.squeeze(-1), batch, dim=0)
        attention_scores = attention_scores.unsqueeze(-1)

        x_weighted = x * attention_scores
        x_mean = global_mean_pool(x_weighted, batch)
        x_max = global_max_pool(x, batch)

        embedding = (x_mean + x_max) / 2

        return embedding

class ExpandedCellEncoder(nn.Module):
    """
    Simplified 2-layer cell encoder.

    Architecture: 1329 → 512 → 256
    - Residual connection at layer 2
    - Skip connection from input to output
    - BatchNorm + ReLU + Dropout per layer
    """
    def __init__(self, input_dim=1329, output_dim=256):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.dropout1 = nn.Dropout(0.4)

        self.fc2 = nn.Linear(512, output_dim)
        self.bn2 = nn.BatchNorm1d(output_dim)
        self.dropout2 = nn.Dropout(0.3)
        self.residual2 = nn.Linear(512, output_dim)

        self.skip = nn.Linear(input_dim, output_dim)

        self.relu = nn.ReLU()

        if torch.cuda.is_available():
            self.device = torch.device('cuda')
        elif torch.backends.mps.is_available():
            self.device = torch.device('mps')
        else:
            self.device = torch.device('cpu')
        self.to(self.device)

    def forward(self, x):
        x = x.to(self.device)

        out1 = self.fc1(x)
        out1 = self.bn1(out1)
        out1 = self.relu(out1)
        out1 = self.dropout1(out1)

        out2 = self.fc2(out1)
        out2 = self.bn2(out2)
        res2 = self.residual2(out1)
        out2 = self.relu(out2 + res2)
        out2 = self.dropout2(out2)

        skip = self.skip(x)
        output = out2 + skip

        return output

CellEncoder = ExpandedCellEncoder

### 3.4 Full Model

In [ ]:
class DrugResponsePathwayGNN(nn.Module):
    def __init__(self, drug_node_dim=9, drug_edge_dim=4, cell_input_dim=1329,
                 n_proteins=0, hidden_dim=256, dropout=0.3):
        super(DrugResponsePathwayGNN, self).__init__()

        if torch.cuda.is_available():
            self.device = torch.device('cuda')
        elif torch.backends.mps.is_available():
            self.device = torch.device('mps')
        else:
            self.device = torch.device('cpu')

        self.n_proteins = n_proteins
        self.hidden_dim = hidden_dim
        self.cell_input_dim = cell_input_dim

        if n_proteins > 0:
            self.protein_proj = nn.Linear(n_proteins, 32)
            self.protein_dim = 32
        else:
            self.protein_proj = None
            self.protein_dim = 0

        self.drug_encoder = DrugEncoder(
            node_feature_dim=drug_node_dim,
            edge_feature_dim=drug_edge_dim,
            hidden_dim=hidden_dim,
            dropout=dropout
        )

        self.cell_encoder = ExpandedCellEncoder(
            input_dim=cell_input_dim,
            output_dim=hidden_dim
        )

        self.attention_query = nn.Linear(hidden_dim, hidden_dim)
        self.attention_key   = nn.Linear(hidden_dim, hidden_dim)
        self.attention_value = nn.Linear(hidden_dim, hidden_dim)

        combined_dim = hidden_dim * 2 + self.protein_dim  # 544 or 512

        self.ic50_head = nn.Sequential(
            nn.Linear(combined_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

        self.classification_head = nn.Sequential(
            nn.Linear(combined_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

        self.reconstruction_head = nn.Sequential(
            nn.Linear(combined_dim, 256),
            nn.ReLU(),
            nn.Linear(256, cell_input_dim)
        )

        self.to(self.device)

    def integrate_with_attention(self, drug_emb, cell_emb):
        query = self.attention_query(drug_emb)
        key = self.attention_key(cell_emb)
        value = self.attention_value(cell_emb)
        attention_scores = torch.matmul(query.unsqueeze(1), key.unsqueeze(2))
        attention_scores = attention_scores / (self.hidden_dim ** 0.5)
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attended_cell = attention_weights.squeeze(-1) * value
        return torch.cat([drug_emb, attended_cell], dim=1)

    def forward(self, drug_batch, cell_batch, proteins=None, has_proteomics=None, return_embeddings=False):
        drug_emb = self.drug_encoder(drug_batch)
        cell_emb = self.cell_encoder(cell_batch.to(self.device))
        combined = self.integrate_with_attention(drug_emb, cell_emb)
        if self.protein_proj is not None:
            if proteins is not None and proteins.numel() > 0:
                prot_emb = self.protein_proj(proteins.to(self.device))
                if has_proteomics is not None:
                    mask = has_proteomics.to(self.device)
                    prot_emb = prot_emb * mask
            else:
                prot_emb = torch.zeros(cell_emb.size(0), self.protein_dim, device=self.device)
            combined = torch.cat([combined, prot_emb], dim=1)
        ic50_pred = self.ic50_head(combined)
        class_pred = self.classification_head(combined)
        recon_pred = self.reconstruction_head(combined)
        results = {
            'ic50': ic50_pred,
            'classification': class_pred,
            'reconstruction': recon_pred
        }
        if return_embeddings:
            results['embeddings'] = {
                'drug': drug_emb,
                'cell': cell_emb,
                'combined': combined
            }
        return results

## 4. Training Framework

Collate functions, pre-batching, evaluation, data loader construction, and the training loop with early stopping and cosine annealing. All functions are used by the LOCO CV pipeline in Section 5.


### 4.1 Data Collation and Pre-batching

In [ ]:
def collate_fn(batch):
    """Custom collate function for drug graphs."""
    drug_graphs = [item['drug_graph'] for item in batch]
    cell_exprs = torch.stack([item['cell_expr'] for item in batch])
    ic50s = torch.stack([item['ic50'] for item in batch])
    drug_names = [item['drug_name'] for item in batch]
    cell_ids = [item['cell_id'] for item in batch]
    drug_batch = Batch.from_data_list(drug_graphs)
    out = {
        'drug_batch': drug_batch,
        'cell_batch': cell_exprs,
        'ic50': ic50s,
        'drug_names': drug_names,
        'cell_ids': cell_ids
    }
    if 'proteins' in batch[0] and batch[0]['proteins'].numel() > 0:
        out['proteins'] = torch.stack([item['proteins'] for item in batch])
        out['has_proteomics'] = torch.stack([item['has_proteomics'] for item in batch])
    return out

def prebatched_collate_fn(batch):
    """
    Collate function for pre-batched data.
    Since batch_size=1, each item is already a complete batch, just return it.
    """
    return batch[0]  # batch_size=1 means batch is a list with one item

class PrebatchedDataset(Dataset):
    """
    Dataset for pre-batched data with epoch-level batch shuffling.
    """
    def __init__(self, prebatched_batches):
        """
        Initialize pre-batched dataset.

        Args:
            prebatched_batches: List of pre-batched dictionaries
        """
        self.batches = prebatched_batches
        self.current_order = list(range(len(self.batches)))

    def __len__(self):
        return len(self.batches)

    def __getitem__(self, idx):
        return self.batches[self.current_order[idx]]

    def shuffle_batches(self, random_seed=None):
        """
        Shuffle batch order for epoch-level randomization.

        Args:
            random_seed: Optional random seed for reproducibility
        """
        if random_seed is not None:
            np.random.seed(random_seed)
        np.random.shuffle(self.current_order)

def create_prebatched_data(dataset, batch_size, split_name, phase_name, output_dir, shuffle_samples=True):
    """
    Pre-compute and save batched data to disk.

    Args:
        dataset: DrugResponsePathwayDataset
        batch_size: Batch size
        split_name: Name of split (train/val/test)
        phase_name: Name of training phase (phase1/phase2/phase3)
        output_dir: Directory to save pre-batched data
        shuffle_samples: Whether to shuffle samples before batching

    Returns:
        Path to saved batch file
    """
    project_root = output_dir.parent if output_dir.name == "results" else output_dir
    prebatched_dir = project_root / "prebatched_data" / "tnbc" / f"{phase_name}_cell_line"
    prebatched_dir.mkdir(parents=True, exist_ok=True)

    batch_file = prebatched_dir / f"{split_name}_batches.pkl"
    # Skip cache for CV fold-specific phases - splits change when data changes
    is_cv_fold = "_fold" in str(phase_name)
    if batch_file.exists() and not is_cv_fold:
        return batch_file

    print(f"Pre-batching {split_name} ({len(dataset)} samples, batch_size={batch_size})")

    indices = np.arange(len(dataset))
    if shuffle_samples:
        np.random.seed(42)
        np.random.shuffle(indices)

    try:
        n_batches = 0
        with open(batch_file, 'wb') as f:
            for i in tqdm(range(0, len(indices), batch_size), desc=f"Pre-batching {split_name}"):
                batch_indices = indices[i:i+batch_size]
                batch_items = [dataset[idx] for idx in batch_indices]

                drug_graphs = [item['drug_graph'] for item in batch_items]
                cell_exprs = torch.stack([item['cell_expr'] for item in batch_items])
                ic50s = torch.stack([item['ic50'] for item in batch_items])
                drug_batch = Batch.from_data_list(drug_graphs)

                batched_data = {
                    'drug_batch': drug_batch,
                    'cell_batch': cell_exprs,
                    'ic50': ic50s
                }
                if batch_items[0].get('proteins') is not None and batch_items[0]['proteins'].numel() > 0:
                    batched_data['proteins'] = torch.stack([item['proteins'] for item in batch_items])
                    batched_data['has_proteomics'] = torch.stack([item['has_proteomics'] for item in batch_items])
                pickle.dump(batched_data, f)
                n_batches += 1

        print(f"  Saved {n_batches} batches")
        return batch_file
    except Exception as e:
        if batch_file.exists():
            batch_file.unlink()
        raise RuntimeError(f"Failed to prebatch data for {split_name} in {phase_name}: {e}") from e

### 4.2 Loss and Evaluation

In [ ]:
def compute_per_drug_median_labels_from_dataloader(dataloader):
    """
    Compute per-drug median labels from a dataloader.

    For each drug, computes the median IC50 across all cell lines in the dataloader.
    Labels: Sensitive (1) if IC50 < drug_median, Resistant (0) if IC50 >= drug_median.

    Args:
        dataloader: DataLoader with batches containing 'drug_names', 'cell_ids', 'ic50'

    Returns:
        Dictionary mapping (drug_name, cell_id) -> label (1=sensitive, 0=resistant)
        Dictionary mapping drug_name -> median_ic50 threshold
    """
    all_drug_names = []
    all_cell_ids = []
    all_ic50s = []

    for batch in dataloader:
        if 'drug_names' in batch and 'cell_ids' in batch:
            all_drug_names.extend(batch['drug_names'])
            all_cell_ids.extend(batch['cell_ids'])
            all_ic50s.append(batch['ic50'].cpu().numpy())

    if len(all_drug_names) == 0 or len(all_ic50s) == 0:
        return {}, {}

    ic50_array = np.concatenate(all_ic50s).flatten()

    # Create DataFrame
    df = pd.DataFrame({
        'DrugName': all_drug_names,
        'ModelID': all_cell_ids,
        'LN_IC50': ic50_array
    })

    # Compute per-drug median thresholds
    drug_medians = df.groupby('DrugName')['LN_IC50'].median().to_dict()

    # Create labels: Sensitive (1) if IC50 < drug_median, Resistant (0) if IC50 >= drug_median
    label_map = {}
    for _, row in df.iterrows():
        drug_name = row['DrugName']
        cell_id = row['ModelID']
        ic50 = row['LN_IC50']
        drug_median = drug_medians[drug_name]

        # Sensitive if IC50 is below median (lower IC50 = more sensitive)
        label = 1 if ic50 < drug_median else 0
        key = (drug_name, cell_id)
        label_map[key] = label

    return label_map, drug_medians

In [ ]:
def compute_loss(model, predictions, targets, drug_names=None, cell_ids=None,
                 drug_thresholds=None, median_ic50=None):
    """
    Compute fixed-weight multi-task loss (Test A: uncertainty weighting removed).

    Uses fixed weights: loss = 1.0 * ic50_loss + 0.3 * classification_loss + 0.3 * reconstruction_loss

    Args:
        model: Model (log_var parameters no longer used)
        predictions: Dictionary with 'ic50', 'classification', 'reconstruction'
        targets: Dictionary with 'ic50', 'cell_batch' (for reconstruction)
        drug_names: Array/tensor of drug names for each sample
        cell_ids: Array/tensor of cell IDs for each sample
        drug_thresholds: Dictionary mapping drug_name -> threshold_value (per-drug median method, preferred)
        median_ic50: Global threshold for classification (fallback)

    Returns:
        Total loss and individual losses
    """
    ic50_pred = predictions['ic50'].squeeze()
    ic50_target = targets['ic50'].squeeze()
    batch_size = targets['ic50'].shape[0]  # Before squeeze; batch=1 gives 0D after squeeze

    huber_loss = nn.HuberLoss(delta=1.0)
    ic50_loss = huber_loss(ic50_pred, ic50_target)

    class_pred = predictions['classification'].squeeze()

    # Use per-drug median thresholds (preferred method)
    if drug_thresholds is not None and drug_names is not None:
        if isinstance(drug_names, torch.Tensor):
            drug_names_list = drug_names.cpu().numpy().tolist()
        else:
            drug_names_list = list(drug_names) if hasattr(drug_names, '__iter__') and not isinstance(drug_names, str) else [drug_names]

        if batch_size == 1:
            drug_name = drug_names_list[0] if isinstance(drug_names_list, list) else drug_names_list
            threshold = drug_thresholds.get(drug_name, median_ic50 if median_ic50 is not None else torch.median(ic50_target))
            if isinstance(threshold, (int, float)):
                threshold = torch.tensor(threshold, device=ic50_target.device, dtype=ic50_target.dtype)
            class_target = (ic50_target > threshold).float()
        else:
            thresholds = torch.zeros_like(ic50_target)
            for i, drug_name in enumerate(drug_names_list):
                threshold = drug_thresholds.get(drug_name, median_ic50 if median_ic50 is not None else torch.median(ic50_target))
                if isinstance(threshold, (int, float)):
                    threshold = torch.tensor(threshold, device=ic50_target.device, dtype=ic50_target.dtype)
                thresholds[i] = threshold
            class_target = (ic50_target > thresholds).float()
    else:
        # Fallback to global median threshold
        if median_ic50 is None:
            median_ic50 = torch.median(ic50_target)
        elif isinstance(median_ic50, (int, float)):
            median_ic50 = torch.tensor(median_ic50, device=ic50_target.device, dtype=ic50_target.dtype)
        class_target = (ic50_target > median_ic50).float()

    bce_loss = nn.BCEWithLogitsLoss()(class_pred, class_target)

    recon_pred = predictions['reconstruction']
    if 'cell_batch' in targets:
        recon_target = targets['cell_batch'].detach()
    else:
        recon_target = None

    if recon_target is not None:
        recon_loss = nn.MSELoss()(recon_pred, recon_target)
    else:
        recon_loss = torch.tensor(0.0, device=ic50_pred.device)

    # Fixed-weight losses (Test A: Remove uncertainty weighting)
    total_loss = 1.0 * ic50_loss + 0.3 * bce_loss + 0.05 * recon_loss

    return total_loss, {
        'ic50_loss': ic50_loss.item(),
        'bce_loss': bce_loss.item(),
        'recon_loss': recon_loss.item(),
    }

def evaluate_model(model, dataloader, device, median_ic50=None, is_prebatched=False):
    """
    Evaluate model on a dataloader.

    Args:
        model: Trained model
        dataloader: DataLoader to evaluate on
        device: Device to run on
        median_ic50: Classification threshold (if None, computed from dataloader)
        is_prebatched: Whether using pre-batched data (batch_size=1)

    Returns:
        Dictionary with metrics, predictions, and targets
    """
    model.eval()
    all_preds = []
    all_targets = []
    total_loss = 0

    dataloader_list = list(dataloader)
    if len(dataloader_list) == 0:
        return {
            'metrics': {
                'loss': 0.0,
                'pearson': np.nan,
                'spearman': np.nan,
                'r2': np.nan,
                'rmse': np.nan,
                'mae': np.nan
            },
            'predictions': np.array([]),
            'targets': np.array([])
        }

    _, drug_thresholds = compute_per_drug_median_labels_from_dataloader(dataloader)

    if len(drug_thresholds) == 0:
        # Fallback: collect IC50s for global median threshold
        all_ic50s = []
        for batch in dataloader_list:
            all_ic50s.append(batch['ic50'].cpu().numpy())

        if len(all_ic50s) == 0:
            return {
                'metrics': {
                    'loss': 0.0,
                    'pearson': np.nan,
                    'spearman': np.nan,
                    'r2': np.nan,
                    'rmse': np.nan,
                    'mae': np.nan
                },
                'predictions': np.array([]),
                'targets': np.array([])
            }

        ic50_array = np.concatenate(all_ic50s)
        if median_ic50 is None:
            median_ic50 = np.median(ic50_array)
        drug_thresholds = None

    with torch.no_grad():
        for batch in dataloader_list:
            # PyG scatter operations are sensitive to device placement - ensure clean move
            drug_batch = batch['drug_batch']
            # If batch is already partially on device, move to CPU first to avoid mixed states
            if hasattr(drug_batch, 'x') and torch.is_tensor(drug_batch.x):
                if drug_batch.x.device.type != 'cpu':
                    drug_batch = drug_batch.cpu()
            drug_batch = drug_batch.to(device)

            cell_batch = batch['cell_batch'].to(device)
            ic50_target = batch['ic50'].to(device)
            batch_drug_names = batch.get('drug_names', None)
            batch_cell_ids = batch.get('cell_ids', None)
            proteins = batch.get('proteins')
            has_proteomics = batch.get('has_proteomics')

            outputs = model(drug_batch, cell_batch, proteins=proteins, has_proteomics=has_proteomics, return_embeddings=True)

            targets = {
                'ic50': ic50_target,
                'cell_batch': cell_batch,
            }

            loss, _ = compute_loss(model, outputs, targets,
                                 drug_names=batch_drug_names, cell_ids=batch_cell_ids,
                                 drug_thresholds=drug_thresholds, median_ic50=median_ic50)
            total_loss += loss.item()

            all_preds.append(outputs['ic50'].cpu().numpy())
            all_targets.append(ic50_target.cpu().numpy())

    if len(all_preds) == 0:
        return {
            'metrics': {
                'loss': 0.0,
                'pearson': np.nan,
                'spearman': np.nan,
                'r2': np.nan,
                'rmse': np.nan,
                'mae': np.nan
            },
            'predictions': np.array([]),
            'targets': np.array([])
        }

    all_preds = np.concatenate(all_preds).flatten()
    all_targets = np.concatenate(all_targets).flatten()

    all_preds = np.asarray(all_preds, dtype=np.float64)
    all_targets = np.asarray(all_targets, dtype=np.float64)

    pearson_r, _ = pearsonr(all_targets, all_preds)
    spearman_r, _ = spearmanr(all_targets, all_preds)
    r2 = r2_score(all_targets, all_preds)
    rmse = np.sqrt(mean_squared_error(all_targets, all_preds))
    mae = mean_absolute_error(all_targets, all_preds)

    return {
        'metrics': {
            'loss': total_loss / len(dataloader_list) if len(dataloader_list) > 0 else 0.0,
            'pearson': pearson_r,
            'spearman': spearman_r,
            'r2': r2,
            'rmse': rmse,
            'mae': mae
        },
        'predictions': all_preds,
        'targets': all_targets
    }

### 4.3 Dataloader Construction

In [ ]:
def create_pathway_dataloaders(dataset, train_idx, val_idx, test_idx, batch_size=128,
                                use_prebatched=False, phase_name=None, output_dir=None):
    """
    Create dataloaders using pre-defined cell-line based splits.

    Args:
        dataset: DrugResponsePathwayDataset
        train_idx: Training indices (from original dataframe)
        val_idx: Validation indices (from original dataframe)
        test_idx: Test indices (from original dataframe)
        batch_size: Batch size
        use_prebatched: Whether to use pre-batched data
        phase_name: Name of training phase (required if use_prebatched=True)
        output_dir: Directory for pre-batched data (required if use_prebatched=True)

    Returns:
        Dictionary with train/val/test loaders and datasets (if pre-batched)
    """

    train_data = dataset.original_data.loc[train_idx].reset_index(drop=True)
    val_data = dataset.original_data.loc[val_idx].reset_index(drop=True)
    test_data = dataset.original_data.loc[test_idx].reset_index(drop=True)

    mutation_cols_param = getattr(dataset, 'mutation_cols', None)
    prot_kw = {}
    if getattr(dataset, 'n_proteins', 0) > 0 and getattr(dataset, 'proteomics_by_modelid', None) is not None:
        train_model_ids = set(train_data['ModelID'].astype(str))
        train_with_prot = [m for m in train_model_ids if m in dataset.proteomics_by_modelid.index]
        cols = [c for c in dataset.protein_cols if c in dataset.proteomics_by_modelid.columns]
        if train_with_prot and len(cols) > 0:
            pt = dataset.proteomics_by_modelid.loc[train_with_prot, cols]
            pm_raw = pt.mean(axis=0).values
            ps_raw = np.where(pt.std(axis=0).values > 1e-8, pt.std(axis=0).values, 1.0)
            pm = np.zeros(dataset.n_proteins, dtype=np.float32)
            ps = np.ones(dataset.n_proteins, dtype=np.float32)
            for i, c in enumerate(dataset.protein_cols):
                if c in cols:
                    j = cols.index(c)
                    pm[i] = pm_raw[j]
                    ps[i] = ps_raw[j]
        else:
            pm = np.zeros(dataset.n_proteins, dtype=np.float32)
            ps = np.ones(dataset.n_proteins, dtype=np.float32)
        prot_kw = dict(proteomics_by_modelid=dataset.proteomics_by_modelid, protein_cols=dataset.protein_cols, protein_mean=pm, protein_std=ps)
    train_dataset = DrugResponsePathwayDataset(train_data, dataset.drug_graphs, dataset.drug_col, pathway_cols=dataset.pathway_cols, mutation_cols=mutation_cols_param, **prot_kw)
    val_dataset = DrugResponsePathwayDataset(val_data, dataset.drug_graphs, dataset.drug_col, pathway_cols=dataset.pathway_cols, mutation_cols=mutation_cols_param, **prot_kw)
    test_dataset = DrugResponsePathwayDataset(test_data, dataset.drug_graphs, dataset.drug_col, pathway_cols=dataset.pathway_cols, mutation_cols=mutation_cols_param, **prot_kw)

    if use_prebatched:
        if phase_name is None or output_dir is None:
            raise ValueError("phase_name and output_dir required when use_prebatched=True")

        train_batch_file = create_prebatched_data(train_dataset, batch_size, 'train', phase_name, output_dir, shuffle_samples=True)
        val_batch_file = create_prebatched_data(val_dataset, batch_size, 'val', phase_name, output_dir, shuffle_samples=False)
        test_batch_file = create_prebatched_data(test_dataset, batch_size, 'test', phase_name, output_dir, shuffle_samples=False)

        def _load_batches(path):
            batches = []
            with open(path, 'rb') as f:
                while True:
                    try:
                        batches.append(pickle.load(f))
                    except EOFError:
                        break
            # Legacy format: single pickle of list
            if len(batches) == 1 and isinstance(batches[0], list):
                return batches[0]
            return batches
        train_batches = _load_batches(train_batch_file)
        val_batches = _load_batches(val_batch_file)
        test_batches = _load_batches(test_batch_file)

        train_prebatched = PrebatchedDataset(train_batches)
        val_prebatched = PrebatchedDataset(val_batches)
        test_prebatched = PrebatchedDataset(test_batches)

        train_loader = DataLoader(
            train_prebatched,
            batch_size=1,
            shuffle=False,
            collate_fn=prebatched_collate_fn,
            num_workers=4 if torch.cuda.is_available() else 0,
            pin_memory=torch.cuda.is_available(),
        )

        val_loader = DataLoader(
            val_prebatched,
            batch_size=1,
            shuffle=False,
            collate_fn=prebatched_collate_fn,
            num_workers=4 if torch.cuda.is_available() else 0,
            pin_memory=torch.cuda.is_available(),
        )

        test_loader = DataLoader(
            test_prebatched,
            batch_size=1,
            shuffle=False,
            collate_fn=prebatched_collate_fn,
            num_workers=4 if torch.cuda.is_available() else 0,
            pin_memory=torch.cuda.is_available(),
        )

        return {
            'train': train_loader,
            'val': val_loader,
            'test': test_loader,
            'datasets': {
                'train': train_prebatched,
                'val': val_prebatched,
                'test': test_prebatched
            }
        }
    else:
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            drop_last=True,
            collate_fn=collate_fn,
            num_workers=4 if torch.cuda.is_available() else 0,
            pin_memory=torch.cuda.is_available(),
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            collate_fn=collate_fn,
            num_workers=4 if torch.cuda.is_available() else 0,
            pin_memory=torch.cuda.is_available(),
        )

        test_loader = DataLoader(
            test_dataset,
            batch_size=batch_size,
            shuffle=False,
            collate_fn=collate_fn,
            num_workers=4 if torch.cuda.is_available() else 0,
            pin_memory=torch.cuda.is_available(),
        )

        return {
            'train': train_loader,
            'val': val_loader,
            'test': test_loader
        }

### 4.4 Training Loop

In [ ]:
def train_phase_pathway(
    model,
    train_loader,
    val_loader,
    device,
    phase_name,
    num_epochs=50,
    lr=1e-3,
    weight_decay=1e-4,
    patience=10,
    scheduler_patience=3,
    scheduler_factor=0.7,
    scheduler_min_lr=1e-6,
    checkpoint_path=None,
    prebatched_datasets=None,
    optimizer=None,
    scheduler=None,
    gradient_clip_max_norm=2.0,
    fine_tuning=False,
    freeze_layers=None,
    freeze_drug_encoder=False,
    use_scheduler=True,
    phase_num=None
):
    """
    Train a phase of the pathway-based model.

    Args:
        model: DrugResponsePathwayGNN model
        train_loader: Training DataLoader
        val_loader: Validation DataLoader
        device: Device to train on
        phase_name: Name of the phase
        num_epochs: Maximum epochs
        lr: Learning rate
        weight_decay: Weight decay
        patience: Early stopping patience
        scheduler_patience: LR scheduler patience
        checkpoint_path: Path to save best model
        prebatched_datasets: Dictionary with 'train' PrebatchedDataset (for epoch-level shuffling)
        optimizer: Optional optimizer (if None, creates new AdamW optimizer)
        scheduler: Optional scheduler (if None, creates new OneCycleLR scheduler)

    Returns:
        Trained model and training history
    """
    model = model.to(device)

    if freeze_drug_encoder and hasattr(model, 'drug_encoder'):
        for param in model.drug_encoder.parameters():
            param.requires_grad = False
        frozen = sum(p.numel() for p in model.drug_encoder.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in model.parameters())

    elif freeze_layers is not None:
        all_layers = list(model.named_parameters())
        n_total = len(all_layers)
        n_freeze = int(n_total * freeze_layers)
        for name, param in all_layers[:n_freeze]:
            param.requires_grad = False
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in model.parameters())

    if optimizer is None:
        params = [p for p in model.parameters() if p.requires_grad] if fine_tuning else model.parameters()
        optimizer = AdamW(params, lr=lr, weight_decay=weight_decay)

    is_prebatched = prebatched_datasets is not None and 'train' in prebatched_datasets

    median_ic50 = None

    _, drug_thresholds = compute_per_drug_median_labels_from_dataloader(train_loader)

    if len(drug_thresholds) == 0:
        # Fallback: collect IC50s for global median threshold
        all_ic50s = []
        for batch in train_loader:
            all_ic50s.append(batch['ic50'].cpu().numpy())

        if len(all_ic50s) == 0:
            raise ValueError("Empty training dataloader!")

        ic50_array = np.concatenate(all_ic50s)
        drug_thresholds = None
        median_ic50 = np.median(ic50_array)
    else:
        all_ic50s = []
        for batch in train_loader:
            all_ic50s.append(batch['ic50'].cpu().numpy())
        ic50_array = np.concatenate(all_ic50s)
        median_ic50 = np.median(ic50_array)

    steps_per_epoch = len(train_loader)
    if scheduler is None and use_scheduler:
        if fine_tuning:
                scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-7)
        else:
            scheduler = OneCycleLR(
                optimizer,
                max_lr=3e-3,
                epochs=num_epochs,
                steps_per_epoch=steps_per_epoch,
                pct_start=0.1,
                anneal_strategy='cos'
            )

    best_val_r2 = float('-inf')
    best_train_loss = float('inf')
    best_val_loss = float('inf')
    best_val_pearson = float('-inf')
    best_val_rmse = float('inf')
    best_epoch = 0
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': [], 'val_r2': [], 'val_pearson': []}

    for epoch in range(num_epochs):
        # Shuffle batches at start of each epoch (for pre-batched data)
        if is_prebatched:
            prebatched_datasets['train'].shuffle_batches(random_seed=epoch)

        model.train()
        train_loss = 0
        for batch_idx, batch in enumerate(train_loader):
            if epoch == 0 and batch_idx == 0:
                current_lr = optimizer.param_groups[0]['lr']
                n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
                n_total = sum(p.numel() for p in model.parameters())
                phase_label = f"Phase {phase_num}" if phase_num is not None else phase_name.split(':')[0] if ':' in phase_name else phase_name

            drug_batch = batch['drug_batch'].to(device)
            cell_batch = batch['cell_batch'].to(device)
            ic50_target = batch['ic50'].to(device)
            proteins = batch.get('proteins')
            has_proteomics = batch.get('has_proteomics')

            outputs = model(drug_batch, cell_batch, proteins=proteins, has_proteomics=has_proteomics, return_embeddings=True)

            targets = {
                'ic50': ic50_target,
                'cell_batch': cell_batch,
            }

            batch_drug_names = batch.get('drug_names', None)
            batch_cell_ids = batch.get('cell_ids', None)
            loss, _ = compute_loss(model, outputs, targets,
                                 drug_names=batch_drug_names, cell_ids=batch_cell_ids,
                                 drug_thresholds=drug_thresholds, median_ic50=median_ic50)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=gradient_clip_max_norm)
            optimizer.step()
            if scheduler is not None and not fine_tuning:
                scheduler.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)
        if scheduler is not None and fine_tuning:
            scheduler.step()

        # Use same drug_thresholds for validation (computed from training data)
        eval_result = evaluate_model(model, val_loader, device, median_ic50=median_ic50,
                                   is_prebatched=is_prebatched)
        val_metrics = eval_result['metrics']
        val_r2 = val_metrics['r2']

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_metrics['loss'])
        history['val_r2'].append(val_r2)
        history['val_pearson'].append(val_metrics['pearson'])

        if val_r2 > best_val_r2:
            best_val_r2 = val_r2
            best_train_loss = train_loss
            best_val_loss = val_metrics['loss']
            best_val_pearson = val_metrics['pearson']
            best_val_rmse = val_metrics['rmse']
            best_epoch = epoch + 1
            patience_counter = 0
            if checkpoint_path:
                torch.save({
                    'model_state_dict': model.state_dict(),
                    'val_metrics': val_metrics,
                    'classification_threshold': median_ic50,
                    'epoch': epoch + 1
                }, checkpoint_path)
        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0 or epoch == num_epochs - 1:
            print(f"  Epoch {epoch+1}/{num_epochs} | best@{best_epoch}: R²={best_val_r2:.4f}, r={best_val_pearson:.4f}, RMSE={best_val_rmse:.4f}")

        if patience_counter >= patience:
            print(f"  Early stop at epoch {epoch+1}")
            break

    if checkpoint_path and Path(checkpoint_path).exists():
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])

    if freeze_drug_encoder and hasattr(model, 'drug_encoder'):
        for param in model.drug_encoder.parameters():
            param.requires_grad = True
    elif freeze_layers is not None:
        for param in model.parameters():
            param.requires_grad = True

    return model, history

## 5. Model Training


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
models_dir = output_dir / "models" / "tnbc"
models_dir.mkdir(parents=True, exist_ok=True)

### 5.1 LOCO Setup

Phase 1 and 2 are trained once with all TNBC cells excluded. Phase 3 is retrained per fold, holding out one TNBC cell line as test. Protein selection is done per fold on train/val cells only to prevent leakage.

In [ ]:
_prebatched_dir = project_root / "prebatched_data" / "tnbc"
if _prebatched_dir.exists():
    shutil.rmtree(_prebatched_dir)

all_tnbc_cells = tnbc_pathway['ModelID'].unique().tolist()
all_tnbc_set   = set(all_tnbc_cells)
n_cells        = len(all_tnbc_cells)
print(f"LOCO CV: {n_cells} folds, {n_proteins_full} proteins available")

# ── Helper functions ──────────────────────────────────────────────────────────
def select_proteins_by_ic50_corr(trainval_cells, proteomics_df, protein_cols, pathway_df, top_n=50, min_cells=8):
    """Select top_n proteins most correlated with mean LN_IC50 on trainval cells only."""
    cell_mean_ic50 = pathway_df[pathway_df['ModelID'].isin(trainval_cells)].groupby('ModelID')['LN_IC50'].mean()
    cells_with_both = [c for c in trainval_cells
                       if c in proteomics_df.index and c in cell_mean_ic50.index]
    cells_with_both = list(dict.fromkeys(cells_with_both))  # deduplicate preserving order
    if len(cells_with_both) < min_cells:
        prot_sub = proteomics_df.loc[cells_with_both, protein_cols] if cells_with_both else proteomics_df[protein_cols]
        return prot_sub.var(axis=0).sort_values(ascending=False).head(top_n).index.tolist()
    correlations = {}
    for prot in protein_cols:
        vals  = proteomics_df.loc[cells_with_both, prot]
        if isinstance(vals, pd.DataFrame):
            vals = vals.iloc[:, 0]  # take first column if duplicates exist
        ic50s = cell_mean_ic50.loc[cells_with_both]
        valid = vals.notna()
        n_valid = int(valid.sum())
        if n_valid >= min_cells:
            r, _ = pearsonr(vals[valid].values, ic50s[valid].values)
            correlations[prot] = abs(r)
    if not correlations:
        return protein_cols[:top_n]
    return pd.Series(correlations).sort_values(ascending=False).head(top_n).index.tolist()

def cell_line_trainval_split(dataframe, train_ratio=0.9, random_seed=42):
    """Split dataframe by cell line into train/val only."""
    np.random.seed(random_seed)
    cell_lines = dataframe['ModelID'].unique()
    shuffled   = np.random.permutation(cell_lines)
    n_train    = max(1, int(len(cell_lines) * train_ratio))
    train_cells = set(shuffled[:n_train])
    val_cells   = set(shuffled[n_train:])
    train_idx = dataframe[dataframe['ModelID'].isin(train_cells)].index.values
    val_idx   = dataframe[dataframe['ModelID'].isin(val_cells)].index.values
    return train_idx, val_idx

def build_prot_kw(fold_protein_cols, fold_proteomics, trainval_cells):
    """Compute per-fold protein z-score stats on trainval cells and return dataset kwargs."""
    if not fold_protein_cols:
        return {}
    trainval_with_prot = [c for c in trainval_cells if c in fold_proteomics.index]
    if trainval_with_prot:
        pt  = fold_proteomics.loc[trainval_with_prot]
        pm  = pt.mean(axis=0).values.astype(np.float32)
        ps  = np.where(pt.std(axis=0).values > 1e-8, pt.std(axis=0).values, 1.0).astype(np.float32)
    else:
        pm  = np.zeros(len(fold_protein_cols), dtype=np.float32)
        ps  = np.ones(len(fold_protein_cols),  dtype=np.float32)
    return dict(proteomics_by_modelid=fold_proteomics, protein_cols=fold_protein_cols,
                protein_mean=pm, protein_std=ps)

### 5.2 Phase 1 & 2: Shared Pre-training

In [ ]:
loco_shared_ckpt_dir = models_dir / "loco_shared"
loco_shared_ckpt_dir.mkdir(exist_ok=True)
p1_shared_ckpt = loco_shared_ckpt_dir / "phase1.pt"
p2_shared_ckpt = loco_shared_ckpt_dir / "phase2.pt"

shared_protein_cols = select_proteins_by_ic50_corr(
    trainval_cells=all_tnbc_cells,
    proteomics_df=proteomics_by_modelid_full,
    protein_cols=protein_cols_full,
    pathway_df=tnbc_pathway,
    top_n=50,
    min_cells=8,
)
shared_n_proteins  = len(shared_protein_cols)
shared_proteomics  = proteomics_by_modelid_full[shared_protein_cols]
shared_prot_kw     = build_prot_kw(shared_protein_cols, shared_proteomics, all_tnbc_cells)

pan_excl_all    = pan_cancer_pathway[~pan_cancer_pathway['ModelID'].isin(all_tnbc_set)]
breast_excl_all = breast_pathway[~breast_pathway['ModelID'].isin(all_tnbc_set)]

pan_train_idx_shared,    pan_val_idx_shared    = cell_line_trainval_split(pan_excl_all,    train_ratio=0.9, random_seed=42)
breast_train_idx_shared, breast_val_idx_shared = cell_line_trainval_split(breast_excl_all, train_ratio=0.9, random_seed=42)

shared_pan_dataset = DrugResponsePathwayDataset(
    pan_cancer_pathway, drug_graphs, drug_col=drug_name_col,
    pathway_cols=pathway_cols, mutation_cols=mutation_cols, **shared_prot_kw)
shared_breast_dataset = DrugResponsePathwayDataset(
    breast_pathway, drug_graphs, drug_col=drug_name_col,
    pathway_cols=pathway_cols, mutation_cols=mutation_cols, **shared_prot_kw)

_dummy_test_idx = tnbc_pathway[tnbc_pathway['ModelID'] == all_tnbc_cells[0]].index.values

phase1_shared_loaders = create_pathway_dataloaders(
    shared_pan_dataset, pan_train_idx_shared, pan_val_idx_shared, _dummy_test_idx,
    batch_size=256, use_prebatched=True,
    phase_name="phase1_shared", output_dir=output_dir)
phase2_shared_loaders = create_pathway_dataloaders(
    shared_breast_dataset, breast_train_idx_shared, breast_val_idx_shared, _dummy_test_idx,
    batch_size=64, use_prebatched=True,
    phase_name="phase2_shared", output_dir=output_dir)

if p1_shared_ckpt.exists():

    model_p1_shared = DrugResponsePathwayGNN(
        cell_input_dim=total_cell_input_dim, n_proteins=shared_n_proteins).to(device)
    model_p1_shared.load_state_dict(
        torch.load(p1_shared_ckpt, map_location=device, weights_only=False)['model_state_dict'])
else:
    print("Training Phase 1 (shared)...")
    model_p1_shared = DrugResponsePathwayGNN(
        cell_input_dim=total_cell_input_dim, n_proteins=shared_n_proteins).to(device)
    model_p1_shared, _ = train_phase_pathway(
        model_p1_shared, phase1_shared_loaders['train'], phase1_shared_loaders['val'], device,
        phase_name="Phase 1: Pan-Cancer (shared)",
        num_epochs=100, lr=1e-3, weight_decay=1e-4,
        patience=25, checkpoint_path=str(p1_shared_ckpt), gradient_clip_max_norm=2.0,
        fine_tuning=False, freeze_layers=None, phase_num=1,
        prebatched_datasets=phase1_shared_loaders.get('datasets', None))

if p2_shared_ckpt.exists():

    model_p2_shared = DrugResponsePathwayGNN(
        cell_input_dim=total_cell_input_dim, n_proteins=shared_n_proteins).to(device)
    model_p2_shared.load_state_dict(
        torch.load(p2_shared_ckpt, map_location=device, weights_only=False)['model_state_dict'])
else:
    print("Training Phase 2 (shared)...")
    model_p2_shared = DrugResponsePathwayGNN(
        cell_input_dim=total_cell_input_dim, n_proteins=shared_n_proteins).to(device)
    model_p2_shared.load_state_dict(
        torch.load(p1_shared_ckpt, map_location=device, weights_only=False)['model_state_dict'])
    model_p2_shared, _ = train_phase_pathway(
        model_p2_shared, phase2_shared_loaders['train'], phase2_shared_loaders['val'], device,
        phase_name="Phase 2: Breast (shared)",
        num_epochs=50, lr=8e-5, weight_decay=5e-5,
        patience=25, checkpoint_path=str(p2_shared_ckpt), gradient_clip_max_norm=1.5,
        fine_tuning=True, freeze_drug_encoder=True, phase_num=2,
        prebatched_datasets=phase2_shared_loaders.get('datasets', None))

print(f"Starting LOCO Phase 3 ({n_cells} folds)")

### 5.3 Phase 3: LOCO Fold Training

In [ ]:
fold_results    = []
best_fold_loaders = None

for fold_idx, test_cell in enumerate(all_tnbc_cells):
    print(f"\nFold {fold_idx+1}/{n_cells}: {test_cell}")

    test_cells    = {test_cell}
    trainval_cells = set(all_tnbc_cells) - test_cells

    fold_protein_cols = select_proteins_by_ic50_corr(
        trainval_cells=list(trainval_cells),
        proteomics_df=proteomics_by_modelid_full,
        protein_cols=protein_cols_full,
        pathway_df=tnbc_pathway,
        top_n=50,
        min_cells=8,
    )
    fold_n_proteins = len(fold_protein_cols)
    fold_proteomics = proteomics_by_modelid_full[fold_protein_cols]
    fold_prot_kw    = build_prot_kw(fold_protein_cols, fold_proteomics, list(trainval_cells))

    test_idx       = tnbc_pathway[tnbc_pathway['ModelID'] == test_cell].index.values
    tnbc_trainval  = tnbc_pathway[tnbc_pathway['ModelID'].isin(trainval_cells)]
    tnbc_train_idx, tnbc_val_idx = cell_line_trainval_split(
        tnbc_trainval, train_ratio=0.8, random_seed=42 + fold_idx)

    fold_tnbc_dataset = DrugResponsePathwayDataset(
        tnbc_pathway, drug_graphs, drug_col=drug_name_col,
        pathway_cols=pathway_cols, mutation_cols=mutation_cols, **fold_prot_kw)

    phase3_loaders = create_pathway_dataloaders(
        fold_tnbc_dataset, tnbc_train_idx, tnbc_val_idx, test_idx,
        batch_size=64, use_prebatched=True,
        phase_name=f"phase3_fold{fold_idx+1}", output_dir=output_dir)

    fold_ckpt_dir = models_dir / f"loco_fold{fold_idx+1}"
    fold_ckpt_dir.mkdir(exist_ok=True)
    p3_ckpt = fold_ckpt_dir / "phase3.pt"

    # Note: shared model has shared_n_proteins; fold model has fold_n_proteins
    # If they differ, reinitialise protein_proj (linear layer) — weights incompatible
    model_p3 = DrugResponsePathwayGNN(
        cell_input_dim=total_cell_input_dim, n_proteins=fold_n_proteins).to(device)

    shared_state = torch.load(p2_shared_ckpt, map_location=device, weights_only=False)['model_state_dict']
    if fold_n_proteins != shared_n_proteins:
        shared_state = {k: v for k, v in shared_state.items() if 'protein_proj' not in k}
        model_p3.load_state_dict(shared_state, strict=False)
    else:
        model_p3.load_state_dict(shared_state, strict=True)

    model_p3, _ = train_phase_pathway(
        model_p3, phase3_loaders['train'], phase3_loaders['val'], device,
        phase_name=f"Phase 3: TNBC - Fold {fold_idx+1}",
        num_epochs=50, lr=1e-4, weight_decay=0.01,
        patience=25, checkpoint_path=str(p3_ckpt), gradient_clip_max_norm=1.5,
        fine_tuning=True, freeze_drug_encoder=True, use_scheduler=False, phase_num=3,
        prebatched_datasets=phase3_loaders.get('datasets', None))

    fold_tnbc_shared_dataset = DrugResponsePathwayDataset(
        tnbc_pathway, drug_graphs, drug_col=drug_name_col,
        pathway_cols=pathway_cols, mutation_cols=mutation_cols, **shared_prot_kw)
    shared_test_loaders = create_pathway_dataloaders(
        fold_tnbc_shared_dataset,
        tnbc_train_idx, tnbc_val_idx, test_idx,
        batch_size=64, use_prebatched=True,
        phase_name=f"phase3_shared_fold{fold_idx+1}", output_dir=output_dir)
    shared_test_loader = shared_test_loaders['test']

    r1 = evaluate_model(model_p1_shared, shared_test_loader, device, is_prebatched=True)
    r2 = evaluate_model(model_p2_shared, shared_test_loader, device, is_prebatched=True)
    r3 = evaluate_model(model_p3,        phase3_loaders['test'], device, is_prebatched=True)

    fold_results.append({
        'fold':       fold_idx + 1,
        'test_cell':  test_cell,
        'n_proteins': fold_n_proteins,
        'phase1':     r1['metrics'],
        'phase2':     r2['metrics'],
        'phase3':     r3['metrics'],
    })

    print(f"  R²: P1={r1['metrics']['r2']:.4f}, P2={r2['metrics']['r2']:.4f}, P3={r3['metrics']['r2']:.4f}")

    if best_fold_loaders is None or r3['metrics']['r2'] >= max(r['phase3']['r2'] for r in fold_results):
        best_fold_loaders = {'tnbc': phase3_loaders}

### 5.4 Results Aggregation

In [ ]:
def ci_stats(vals):
    vals = [v for v in vals if not np.isnan(v)]
    if len(vals) < 2:
        m = np.mean(vals) if vals else np.nan
        return m, np.nan, m, m
    mean, std = np.mean(vals), np.std(vals)
    sem = std / np.sqrt(len(vals))
    lo, hi = stats.t.interval(0.95, len(vals) - 1, loc=mean, scale=sem)
    return mean, std, lo, hi

metric_names = ['pearson', 'spearman', 'r2', 'rmse', 'mae']
print(f"\nLOCO CV results ({n_cells} folds):")
for label, key in [('Phase 1', 'phase1'), ('Phase 2', 'phase2'), ('Phase 3', 'phase3')]:
    print(f"  {label}:")
    for m in metric_names:
        vals = [r[key][m] for r in fold_results]
        mean, std, lo, hi = ci_stats(vals)
        print(f"    {m.upper():10s}: {mean:.4f} ± {std:.4f}  [{lo:.4f}, {hi:.4f}]")

for r in fold_results:
    delta = r['phase3']['r2'] - r['phase2']['r2']

loco_results_path = output_dir / "loco_cv_results_cell_line.pkl"
with open(loco_results_path, 'wb') as f:
    pickle.dump({'fold_results': fold_results, 'n_folds': n_cells,
                 'cv_strategy': 'loco', 'cancer': 'tnbc',
                 'shared_p1_ckpt': str(p1_shared_ckpt),
                 'shared_p2_ckpt': str(p2_shared_ckpt)}, f)

best_fold_idx = np.nanargmax([r['phase3']['r2'] for r in fold_results])
best_fold_num = fold_results[best_fold_idx]['fold']

model_phase1 = model_p1_shared
model_phase2 = model_p2_shared
model_phase3 = DrugResponsePathwayGNN(
    cell_input_dim=total_cell_input_dim,
    n_proteins=fold_results[best_fold_idx]['n_proteins']).to(device)
model_phase3.load_state_dict(torch.load(
    models_dir / f"loco_fold{best_fold_num}" / "phase3.pt",
    map_location=device, weights_only=False)['model_state_dict'])

phase1_checkpoint = p1_shared_ckpt
phase2_checkpoint = p2_shared_ckpt
phase3_checkpoint = models_dir / f"loco_fold{best_fold_num}" / "phase3.pt"
tnbc_loaders      = best_fold_loaders['tnbc']

print(f"Best fold: {best_fold_num} ({fold_results[best_fold_idx]['test_cell']}), P3 R²={fold_results[best_fold_idx]['phase3']['r2']:.4f}")